# Asynchronous job arrays

Real workloads submit many jobs at once and collect them later. The asynchronous API answers `202 Accepted` at once,
stores each job durably, and lets the client long-poll for results. This notebook runs a job array over every twin,
lists and filters jobs, and compares the wall time with sending the same jobs one by one.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:9e791f',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. Submit a job array

Three circuits on all seven twins: 21 jobs, accepted in a fraction of a second.

In [2]:
import time, httpx
circuits = {"bell": bell(), "ghz5": ghz(5), "qft4": qft_roundtrip(4, 5)}
t0 = time.perf_counter()
array = [(name, b, submit(b, prog, noise=True, routing=True, shots=2000)["job_id"])
         for name, prog in circuits.items() for b in ALL_BACKENDS]
print(f"{len(array)} jobs accepted in {time.perf_counter() - t0:.2f} s")
print(queues())

21 jobs accepted in 0.10 s
{'google_willow': {'queued': 0, 'running': 1, 'capacity': 1}, 'ibm_heron': {'queued': 0, 'running': 1, 'capacity': 1}, 'ibm_nighthawk': {'queued': 0, 'running': 1, 'capacity': 1}, 'ionq_forte': {'queued': 0, 'running': 1, 'capacity': 1}, 'iqm_emerald': {'queued': 0, 'running': 1, 'capacity': 1}, 'quantinuum_helios': {'queued': 0, 'running': 0, 'capacity': 1}, 'rigetti_ankaa_3': {'queued': 0, 'running': 1, 'capacity': 1}}


## 2. Collect with long polling

`GET /v1/jobs/{id}?wait=s` holds the request open until the job finishes (up to 30 s), so the client does not spin.

In [3]:
t0 = time.perf_counter()
done = {}
for name, b, jid in array:
    body = job(jid, wait=20)
    while body["status"] not in ("COMPLETED", "FAILED", "CANCELLED", "REJECTED"):
        body = job(jid, wait=20)
    done[(name, b)] = body
print(f"all {len(done)} collected {time.perf_counter() - t0:.2f} s later; statuses:",
      pd.Series([d["status"] for d in done.values()]).value_counts().to_dict())
pd.DataFrame([{"circuit": n, "twin": NAMES[b], "top outcome": max(d["result"]["counts"], key=d["result"]["counts"].get)}
              for (n, b), d in done.items()]).pivot(index="circuit", columns="twin", values="top outcome")

all 21 collected 0.10 s later; statuses: {'COMPLETED': 21}


twin,Ankaa-3,Emerald,Forte,Helios,Heron r3,Nighthawk,Willow
circuit,,,,,,,
bell,00,00,00,00,00,00,00
ghz5,00000,00000,00000,00000,00000,00000,00000
qft4,0101,0101,0101,0101,0101,0101,0101


## 3. Listing and filtering jobs

`GET /v1/jobs` lists stored jobs, filtered by status and backend, with the queue statistics of every twin.

In [4]:
listing = httpx.get(f"{API_URL}/v1/jobs", params={"status": "COMPLETED", "backend": "ibm_heron", "limit": 5}).json()
print("keys:", list(listing))
pd.DataFrame(listing["jobs"])[["job_id", "backend", "status", "priority", "submitted_at"]]

keys: ['jobs', 'queues']


,job_id,backend,status,priority,submitted_at
0,bcbff5016449458f8747aeaae53419c9,ibm_heron,COMPLETED,5,2026-11-10T05:58:14.984Z
1,caefa0d73c4b47518bf95ad31291a058,ibm_heron,COMPLETED,5,2026-11-10T05:58:15.011Z
2,c1d7e31b72d64119a8be77c3200c35c8,ibm_heron,COMPLETED,5,2026-11-10T05:59:13.704Z
3,8f91bc1c8ac94d86845cbc93029c8a53,ibm_heron,COMPLETED,5,2026-11-10T05:59:13.728Z
4,314c6fb2458c418b977e9628684029cd,ibm_heron,COMPLETED,5,2026-11-10T05:59:13.762Z


## 4. Asynchronous against one by one

Sent one by one, each job waits for the previous one. Submitted as an array, the jobs run in parallel on the seven twins.

In [5]:
jobs = [(b, ghz(6)) for b in ALL_BACKENDS] * 3
t0 = time.perf_counter()
for b, prog in jobs:
    run(b, prog, noise=True, routing=True, shots=4000)
sequential = time.perf_counter() - t0
t0 = time.perf_counter()
ids = [submit(b, prog, noise=True, routing=True, shots=4000)["job_id"] for b, prog in jobs]
for jid in ids:
    while job(jid, wait=20)["status"] != "COMPLETED":
        pass
parallel = time.perf_counter() - t0
print(f"{len(jobs)} jobs: one by one {sequential:.2f} s, as an asynchronous array {parallel:.2f} s ({sequential / parallel:.1f}x)")

21 jobs: one by one 0.38 s, as an asynchronous array 0.15 s (2.5x)


## 5. Where an asynchronous job waited

An asynchronous job records when it was submitted, started and finished, so its time in the queue is separate from its run time.

In [6]:
body = done[("ghz5", "ibm_heron")]
print({k: body[k] for k in ("submitted_at", "started_at", "finished_at") if k in body})
tel = body["telemetry"]
print("execution mode:", tel["execution_mode"], "| queue clock:", tel["queue_clock"])

{'submitted_at': '2026-11-10T05:59:13.728Z', 'started_at': '2026-11-10T05:59:13.732Z', 'finished_at': '2026-11-10T05:59:13.753Z'}
execution mode: async | queue clock: perf_counter


## Takeaways

- Asynchronous submission returns at once; long polling collects results without busy waiting.
- A job array uses every twin in parallel.